### SMA Training Notebook

This notebook contains code to train regression models on curated SMA price and RSI data.

In [2]:
import tensorflow as tf
import numpy as np
import sklearn as sk
import os

I0000 00:00:1776622730.439587    8240 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776622730.472539    8240 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776622731.403203    8240 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [3]:
import matplotlib.pyplot as plt

FIGURE_FOLDER = 'figures/'

def plot_acc_and_loss(history, title: str, save: bool = False):
    file_title = title.replace(' ', '-').lower()

    if save and not os.path.exists(FIGURE_FOLDER): os.mkdir(FIGURE_FOLDER)

    plt.title("Model and Validation MAE for " + title)
    xs = list(range(1, len(history.history['mae']) + 1))
    plt.plot(xs, history.history['mae'], label="Model MAE", color="Red")
    plt.plot(xs, history.history['val_mae'], label="Validation MAE", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("MAE")
    plt.legend()

    if save: 
        plt.savefig(FIGURE_FOLDER + file_title + '-mae.png')
        plt.close()
    else: plt.show()

    plt.title("Model and Validation Loss for " + title)
    xs = list(range(1, len(history.history['val_loss']) + 1))
    plt.plot(xs, history.history['loss'], label="Model loss", color="Red")
    plt.plot(xs, history.history['val_loss'], label="Validation loss", color="Blue")
    plt.xlabel("Epoch")
    plt.ylabel("Cost")
    plt.legend()
    
    if save: 
        plt.savefig(FIGURE_FOLDER + file_title + '-loss.png')
        plt.close()
    else: plt.show()

def prcnt_within_tolerance(X, Y, tolerance):
    return np.count_nonzero(np.abs(X - Y) <= tolerance) / X.shape[0]

def r2_score(model, X, Y):
    return sk.metrics.r2_score(Y, model.predict(X))

# Difference Formula for real numbers
def mean_prcnt_diff(model, X, Y):
    abs_Y = np.abs(Y)
    abs_Pred = np.abs(model.predict(X))

    raw_diffs = (np.abs(abs_Pred - abs_Y) / ((abs_Pred + abs_Y) / 2))

    #print(raw_diffs[:10])

    return np.mean(raw_diffs[np.isfinite(raw_diffs)])

In [4]:
RESULTS_FOLDER = 'results/'

def train_sma_mse(window: int, offset: int, log: bool = False):
    file = f'stock_sma_{window}_{offset}_curated.csv'

    data = np.loadtxt(file, delimiter=',')

    data = data.T
    np.random.shuffle(data)
    data = data.T

    X = data[:504,:]
    Y = data[-1:,:]

    y_lo = np.percentile(Y, 2)
    y_hi = np.percentile(Y, 98)
    
    if log: print(f"Clipping Y to [{y_lo:.4f}, {y_hi:.4f}]  (was [{Y.min():.4f}, {Y.max():.4f}])")
    
    Y = np.clip(Y, y_lo, y_hi)

    train_ratio = 0.8
    train_size = int(X.shape[1] * train_ratio)

    X_train = X[:,:train_size].T
    Y_train = Y[:,:train_size].T

    X_val = X[:,train_size:].T
    Y_val = Y[:,train_size:].T

    modelMSE = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

    modelMSE.compile(
        optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae']
    )

    if log: modelMSE.summary()

    modelMSE.fit(X_train, Y_train,
        validation_data=(X_val, Y_val),
        batch_size=64, 
        epochs=30,
        verbose=log,
        shuffle=True)
    
    plot_acc_and_loss(modelMSE.history, f"MSE {window} {offset} Model", save=True)

    y_std = Y.std()
    y_mean = np.full(X_val.shape[0], Y.mean())
    y_mean = y_mean.reshape((X_val.shape[0], 1))

    val_preds = modelMSE.predict(X_val)

    very_close = prcnt_within_tolerance(val_preds, Y_val, y_std / 8) / prcnt_within_tolerance(y_mean, Y_val, y_std / 8)
    close = prcnt_within_tolerance(val_preds, Y_val, y_std / 4) / prcnt_within_tolerance(y_mean, Y_val, y_std / 4)
    moderate = prcnt_within_tolerance(val_preds, Y_val, y_std / 2) / prcnt_within_tolerance(y_mean, Y_val, y_std / 2)
    far = prcnt_within_tolerance(val_preds, Y_val, y_std) / prcnt_within_tolerance(y_mean, Y_val, y_std)
    val_r2 = r2_score(modelMSE, X_val, Y_val)
    total_r2 = r2_score(modelMSE, X.T, Y.T)
    prcnts_off = mean_prcnt_diff(modelMSE, X_val, Y_val)
    
    if not os.path.exists(RESULTS_FOLDER): os.mkdir(RESULTS_FOLDER)

    file_title = f'mse-{window}-{offset}-results.csv'

    with open(RESULTS_FOLDER + file_title, 'w') as f:
        f.write('very_close,close,moderate,far,val_r2,total_r2,prcnts_off\n')
        f.write(f'{very_close},{close},{moderate},{far},{val_r2},{total_r2},{prcnts_off}\n')

    return [ modelMSE, X_train, Y_train, X_val, Y_val ]


In [ ]:
train_sma_mse(3, 3)

In [ ]:
# Note: Running this cell requires 36 data sets and generates 36 models
# This may take hours to run
interval_list = [3,7,15,31,63,126]

for window in interval_list:
    for offset_len in interval_list:
        train_sma_mse(window, offset_len)
        print(f"Completed window:{window}, offset:{offset_len}")

In [5]:
RESULTS_FOLDER = 'results/'

def train_and_eval(label: str, X, Y, model, log: bool = False):
    y_lo = np.percentile(Y, 2)
    y_hi = np.percentile(Y, 98)
    
    if log: print(f"Clipping Y to [{y_lo:.4f}, {y_hi:.4f}]  (was [{Y.min():.4f}, {Y.max():.4f}])")
    
    Y = np.clip(Y, y_lo, y_hi)

    train_ratio = 0.8
    train_size = int(X.shape[1] * train_ratio)

    X_train = X[:,:train_size].T
    Y_train = Y[:,:train_size].T

    X_val = X[:,train_size:].T
    Y_val = Y[:,train_size:].T

    model.fit(X_train, Y_train,
        validation_data=(X_val, Y_val),
        batch_size=64, 
        epochs=30,
        verbose=log,
        shuffle=True)
    
    plot_acc_and_loss(model.history, label, save=True)

    y_std = Y.std()
    y_mean = np.full(X_val.shape[0], Y.mean())
    y_mean = y_mean.reshape((X_val.shape[0], 1))

    val_preds = model.predict(X_val)

    very_close = prcnt_within_tolerance(val_preds, Y_val, y_std / 8) / prcnt_within_tolerance(y_mean, Y_val, y_std / 8)
    close = prcnt_within_tolerance(val_preds, Y_val, y_std / 4) / prcnt_within_tolerance(y_mean, Y_val, y_std / 4)
    moderate = prcnt_within_tolerance(val_preds, Y_val, y_std / 2) / prcnt_within_tolerance(y_mean, Y_val, y_std / 2)
    far = prcnt_within_tolerance(val_preds, Y_val, y_std) / prcnt_within_tolerance(y_mean, Y_val, y_std)
    val_r2 = r2_score(model, X_val, Y_val)
    total_r2 = r2_score(model, X.T, Y.T)
    prcnts_off = mean_prcnt_diff(model, X_val, Y_val)
    
    if not os.path.exists(RESULTS_FOLDER): os.mkdir(RESULTS_FOLDER)

    file_title = f'{label}-results.csv'

    with open(RESULTS_FOLDER + file_title, 'w') as f:
        f.write('very_close,close,moderate,far,val_r2,total_r2,prcnts_off\n')
        f.write(f'{very_close},{close},{moderate},{far},{val_r2},{total_r2},{prcnts_off}\n')

    return [ model, X_train, Y_train, X_val, Y_val ]


In [6]:
data = np.loadtxt('stock_sma_3_3_curated.csv', delimiter=',')

data = data.T
np.random.shuffle(data)
data = data.T

print(data.shape)

X = data[:504,:]
Y = data[-1:,:]

(505, 99451)


In [9]:
modelMSEtanhrelu = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='tanh', kernel_initializer='glorot_uniform', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(504, activation='tanh', kernel_initializer='glorot_uniform', kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.BatchNormalization(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSEtanhrelu.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
train_and_eval("MSE 3 3 Tanh and Relu Model", X, Y, modelMSEtanhrelu)

622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 722us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 690us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 2s 668us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 663us/step


[<Sequential name=sequential, built=True>,
 array([[-1.48069415e+00, -1.49508215e+00, -1.56701863e+00, ...,
          2.32327174e+01,  8.49160330e+01,  8.81353222e+01],
        [ 1.07487442e+00,  1.07928714e+00,  1.09958592e+00, ...,
          9.19548756e+01,  6.61623239e+01,  6.93828440e+01],
        [ 3.94960477e+00,  3.77300122e+00,  3.54288122e+00, ...,
          1.34364394e+01,  9.20594679e+00,  5.54714927e+01],
        ...,
        [ 7.12008327e-03, -4.33961116e-02, -9.85040979e-02, ...,
          5.97843120e+01,  6.81396493e+01,  7.74913910e+01],
        [ 9.40904698e-01,  9.79546078e-01,  9.12666543e-01, ...,
          6.79861126e+01,  4.42255885e+01,  4.96160251e+01],
        [-2.12427870e+00, -2.10230099e+00, -2.15653144e+00, ...,
          5.68390600e+01,  4.65164833e+01,  2.09665393e+01]],
       shape=(79560, 504)),
 array([[-0.20990195],
        [-2.42887857],
        [-0.39056238],
        ...,
        [ 1.94954935],
        [-0.31239838],
        [ 0.4059984 ]], shape=(

In [56]:
modelMSEthin = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSEthin.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [57]:
train_and_eval("MSE 3 3 Thin", X, Y, modelMSEthin)

622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 448us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 419us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 1s 405us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 0s 422us/step


[<Sequential name=sequential_31, built=True>,
 array([[ 0.91076128,  0.97294245,  0.96517259, ..., 30.55358397,
         20.40681775, 10.22167194],
        [-2.0240401 , -2.03565823, -2.0395312 , ..., 80.13237029,
         85.31223875, 90.02726404],
        [ 1.93630786,  1.8338107 ,  1.61639547, ..., 40.19794379,
         34.59351253, 33.2043244 ],
        ...,
        [-0.24616906, -0.44219657, -0.61758609, ..., 82.52191287,
         91.73515355, 75.46381173],
        [-1.73170789, -1.65958208, -1.59251682, ..., 58.11009859,
         94.73690329, 77.32330296],
        [ 1.32888247,  1.51869655,  1.57886071, ..., 70.07326056,
         57.28094156, 47.82296212]], shape=(79560, 504)),
 array([[ 1.42377923],
        [ 1.3125312 ],
        [-0.826646  ],
        ...,
        [ 2.55275487],
        [ 1.80321033],
        [-0.83448815]], shape=(79560, 1)),
 array([[-2.15649236, -2.11114317, -2.09838887, ...,  5.73382174,
         36.6126534 , 40.85089011],
        [-1.07817087, -1.1291932 ,

In [12]:
modelMSEthinconv = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Reshape((504,1)),
        tf.keras.layers.Conv1D(filters=32, kernel_size=10, padding='same', activation='linear'),
        tf.keras.layers.Conv1D(filters=10, kernel_size=3, padding='same', activation='linear'),
        tf.keras.layers.Flatten(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSEthinconv.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
train_and_eval("MSE Thin Conv", X, Y, modelMSEthinconv)

622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 944us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 866us/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 3s 845us/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 843us/step


[<Sequential name=sequential_1, built=True>,
 array([[-1.48069415e+00, -1.49508215e+00, -1.56701863e+00, ...,
          2.32327174e+01,  8.49160330e+01,  8.81353222e+01],
        [ 1.07487442e+00,  1.07928714e+00,  1.09958592e+00, ...,
          9.19548756e+01,  6.61623239e+01,  6.93828440e+01],
        [ 3.94960477e+00,  3.77300122e+00,  3.54288122e+00, ...,
          1.34364394e+01,  9.20594679e+00,  5.54714927e+01],
        ...,
        [ 7.12008327e-03, -4.33961116e-02, -9.85040979e-02, ...,
          5.97843120e+01,  6.81396493e+01,  7.74913910e+01],
        [ 9.40904698e-01,  9.79546078e-01,  9.12666543e-01, ...,
          6.79861126e+01,  4.42255885e+01,  4.96160251e+01],
        [-2.12427870e+00, -2.10230099e+00, -2.15653144e+00, ...,
          5.68390600e+01,  4.65164833e+01,  2.09665393e+01]],
       shape=(79560, 504)),
 array([[-0.20990195],
        [-2.42887857],
        [-0.39056238],
        ...,
        [ 1.94954935],
        [-0.31239838],
        [ 0.4059984 ]], shape

In [14]:
modelMSEthinconvpool = tf.keras.Sequential([
        tf.keras.layers.Dense(504, activation='linear', kernel_regularizer=tf.keras.regularizers.l2(1e-5), input_shape=(504,)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Reshape((504,1)),
        tf.keras.layers.Conv1D(filters=32, kernel_size=10, padding='same', activation='linear'),
        tf.keras.layers.Conv1D(filters=10, kernel_size=3, padding='same', activation='linear'),
        tf.keras.layers.MaxPool1D(pool_size=10),
        tf.keras.layers.Flatten(),
        #tf.keras.layers.Dense(252, activation='tanh', kernel_initializer="glorot_uniform", kernel_regularizer=tf.keras.regularizers.l2(1e-5)),
        tf.keras.layers.Dense(1, activation='linear'),
    ])

modelMSEthinconvpool.compile(optimizer='adam',
        loss=tf.keras.losses.MeanSquaredError(),
        metrics=['mae'])

/home/aus/Code/school/3ML3-Final/.venv/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
train_and_eval("MSE Thin Conv and Pool", X, Y, modelMSEthinconvpool)

622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
3108/3108 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
622/622 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


[<Sequential name=sequential_2, built=True>,
 array([[-1.48069415e+00, -1.49508215e+00, -1.56701863e+00, ...,
          2.32327174e+01,  8.49160330e+01,  8.81353222e+01],
        [ 1.07487442e+00,  1.07928714e+00,  1.09958592e+00, ...,
          9.19548756e+01,  6.61623239e+01,  6.93828440e+01],
        [ 3.94960477e+00,  3.77300122e+00,  3.54288122e+00, ...,
          1.34364394e+01,  9.20594679e+00,  5.54714927e+01],
        ...,
        [ 7.12008327e-03, -4.33961116e-02, -9.85040979e-02, ...,
          5.97843120e+01,  6.81396493e+01,  7.74913910e+01],
        [ 9.40904698e-01,  9.79546078e-01,  9.12666543e-01, ...,
          6.79861126e+01,  4.42255885e+01,  4.96160251e+01],
        [-2.12427870e+00, -2.10230099e+00, -2.15653144e+00, ...,
          5.68390600e+01,  4.65164833e+01,  2.09665393e+01]],
       shape=(79560, 504)),
 array([[-0.20990195],
        [-2.42887857],
        [-0.39056238],
        ...,
        [ 1.94954935],
        [-0.31239838],
        [ 0.4059984 ]], shape